In [2]:
import pandas as pd
import numpy as np

df_droid = pd.read_json('droid_results.jsonl', lines=True)
df_raft = pd.read_json('raft_results.jsonl', lines=True)

df_combined = pd.merge(df_droid, df_raft, on='folder')

df_combined['combined_score'] = np.sqrt(df_combined['droid_score'] * df_combined['raft_score'])

df_combined.to_json('combined_results.jsonl', orient='records', lines=True)

In [31]:
df_combined.to_csv('combined_results.csv', index=False)

In [3]:
print(df_combined[['folder', 'combined_score']].head())

              folder  combined_score
0  lift_eggplant_s42        0.144042
1  lift_eggplant_s43        0.176370
2  lift_eggplant_s44        0.256808
3  lift_eggplant_s45        0.251818
4  lift_eggplant_s46        0.200646


In [4]:
print(df_combined.sort_values(by='combined_score', ascending=False).head())

                       folder  droid_score  raft_score  combined_score
51  put_blue_cup_on_plate_s63     0.799080    0.737066        0.767447
60    put_carrot_on_plate_s42     0.289244    1.934555        0.748036
42  put_blue_cup_on_plate_s54     0.352650    1.553561        0.740178
81    put_carrot_on_plate_s63     0.472375    1.090889        0.717850
49  put_blue_cup_on_plate_s61     0.288237    1.594310        0.677893


In [5]:
df_combined['task'] = df_combined['folder'].str.extract(r'^(.*)_s\d+')
df_sorted = df_combined.sort_values(by=['task', 'combined_score'], ascending=[True, False])
df_top10 = df_sorted.groupby('task').head(10)
print(df_top10)

                             folder  droid_score  raft_score  combined_score  \
9                 lift_eggplant_s51     0.837242    0.323006        0.520033   
11                lift_eggplant_s53     0.444649    0.347032        0.392820   
29                lift_eggplant_s71     0.351494    0.351585        0.351539   
10                lift_eggplant_s52     0.426511    0.272458        0.340891   
21                lift_eggplant_s63     0.296184    0.293348        0.294763   
12                lift_eggplant_s54     0.193413    0.432842        0.289340   
18                lift_eggplant_s60     0.233855    0.357462        0.289127   
8                 lift_eggplant_s50     0.256419    0.305176        0.279737   
28                lift_eggplant_s70     0.195943    0.394805        0.278135   
2                 lift_eggplant_s44     0.193749    0.340390        0.256808   
51        put_blue_cup_on_plate_s63     0.799080    0.737066        0.767447   
42        put_blue_cup_on_plate_s54     

In [6]:
df_top10.to_json('combined_results_top10.jsonl', orient='records', lines=True)
df_top10.to_csv('combined_results_top10.csv', index=False)

In [7]:
summary = df_combined.groupby('task')['combined_score'].agg(['count', 'mean', 'max']).reset_index()
print(summary)

                         task  count      mean       max
0               lift_eggplant     30  0.249066  0.520033
1       put_blue_cup_on_plate     30  0.431045  0.767447
2         put_carrot_on_plate     30  0.414632  0.748036
3       put_eggplant_into_pot     30  0.254550  0.447391
4  stack_blue_cup_on_pink_cup     30  0.362707  0.635885


In [42]:
summary = df_combined.groupby('task')['droid_score'].agg(['count', 'mean', 'max']).reset_index()
print(summary)

                         task  count      mean       max
0               lift_eggplant     30  0.219628  0.837242
1       put_blue_cup_on_plate     30  0.247462  0.799080
2         put_carrot_on_plate     30  0.188627  0.472375
3       put_eggplant_into_pot     30  0.176745  0.528958
4  stack_blue_cup_on_pink_cup     30  0.231731  0.760439


In [43]:
summary = df_combined['combined_score'].agg(['count', 'mean', 'max']).reset_index()
print(summary)

   index  combined_score
0  count      150.000000
1   mean        0.342400
2    max        0.767447


In [10]:
df_low10 = df_sorted.groupby('task').tail(10)
print(df_low10)

                             folder  droid_score  raft_score  combined_score  \
17                lift_eggplant_s59     0.169407    0.248599        0.205218   
27                lift_eggplant_s69     0.137634    0.301621        0.203748   
4                 lift_eggplant_s46     0.131486    0.306183        0.200646   
13                lift_eggplant_s55     0.148835    0.267060        0.199369   
25                lift_eggplant_s67     0.141980    0.266378        0.194474   
26                lift_eggplant_s68     0.131825    0.284561        0.193681   
23                lift_eggplant_s65     0.159950    0.218777        0.187065   
16                lift_eggplant_s58     0.130546    0.246555        0.179407   
1                 lift_eggplant_s43     0.110947    0.280372        0.176370   
0                 lift_eggplant_s42     0.095967    0.216199        0.144042   
35        put_blue_cup_on_plate_s47     0.191029    0.644425        0.350861   
34        put_blue_cup_on_plate_s46     

In [12]:
df_low10.to_json('combined_results_low10.jsonl', orient='records', lines=True)
df_low10.to_csv('combined_results_low10.csv', index=False)

In [28]:
import pandas as pd
import shutil
import os

def copy_videos(df, subdir, video_extension='.mp4'):
    source_dir = '../world-model-eval/rollouts/openvla_seeds'
    target_dir = source_dir + "/" + subdir
    
    os.makedirs(target_dir, exist_ok=True)

    copy_count = 0
    for _, row in df.iterrows():
        folder_name = row['folder']
        score = round(row['combined_score'], 4)
        score_str = str(round(row['combined_score'], 4)).replace('.', '_')
        
        matched_files = [
            f for f in os.listdir(source_dir)
            if f == folder_name + video_extension
        ]
        
        if matched_files:
            for f in matched_files:
                src_file_path = os.path.join(source_dir, f)
                dst_file_name = f"{folder_name}_{score_str}{video_extension}"
                dst_file_path = os.path.join(target_dir, dst_file_name)
                shutil.copy2(src_file_path, dst_file_path)
                copy_count += 1
        else:
            print(f"파일 없음: {folder_name}")

    print(f"[{target_dir}] 총 {copy_count}개의 영상 추출 완료!")

In [29]:
copy_videos(df_top10, 'top_anomaly')

[../world-model-eval/rollouts/openvla_seeds/top_anomaly] 총 50개의 영상 추출 완료!


In [30]:
copy_videos(df_low10, 'low_anomaly')

[../world-model-eval/rollouts/openvla_seeds/low_anomaly] 총 50개의 영상 추출 완료!


# 실물 실험 영상 score 계산

In [38]:
df_droid_real = pd.read_json('droid_results_real.jsonl', lines=True)
df_raft_real = pd.read_json('raft_results_real.jsonl', lines=True)

df_combined_real = pd.merge(df_droid_real, df_raft_real, on='folder')

df_combined_real['combined_score'] = np.sqrt(df_combined_real['droid_score'] * df_combined_real['raft_score'])

df_combined_real.to_json('combined_results_real.jsonl', orient='records', lines=True)
df_combined_real.to_csv('combined_results_real.csv', index=False)

In [39]:
summary = df_combined_real['combined_score'].agg(['count', 'mean', 'max']).reset_index()
print(summary)

   index  combined_score
0  count       30.000000
1   mean        0.267952
2    max        0.672948


In [41]:
summary = df_combined_real['droid_score'].agg(['count', 'mean', 'max']).reset_index()
print(summary)

   index  droid_score
0  count    30.000000
1   mean     0.158861
2    max     0.475736


In [40]:
print(df_combined_real.sort_values(by='combined_score', ascending=False).head())

                                     folder  droid_score  raft_score  \
14                     openvla--lift_cheese     0.219168    2.066266   
29      openvla--stack_blue_cup_on_pink_cup     0.475736    0.840097   
27  openvla--put_eggplant_into_pot--clutter     0.233252    1.522440   
16          openvla--move_banana_near_plate     0.288348    1.070630   
26      openvla--put_corn_on_plate--clutter     0.171104    1.309741   

    combined_score  
14        0.672948  
29        0.632190  
27        0.595913  
16        0.555620  
26        0.473394  
